In [ ]:
import duckdb
from modules.datasets import DATASETS

ISLR = DATASETS["ISLR"]
islr_str = str(ISLR).replace("\\", "/")


db = duckdb.connect().execute(f"""
    CREATE VIEW meta_holistic AS
    SELECT * EXCLUDE (h.path, m.path)
    FROM (
        SELECT * EXCLUDE (filename), ltrim(replace(replace(filename, '\\', '/'), '{islr_str}', ''), '/') AS path,
        FROM read_parquet('{str(ISLR / "**" / "*.parquet")}', filename=true)
    ) h, (
        SELECT * FROM read_csv_auto('{ISLR / "train.csv"}')
    ) m
    WHERE h.path = m.path
""")

db.sql("DESCRIBE SELECT * FROM meta_holistic").show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ frame          │ SMALLINT    │ YES     │ NULL    │ NULL    │ NULL    │
│ row_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ type           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ landmark_index │ SMALLINT    │ YES     │ NULL    │ NULL    │ NULL    │
│ x              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ y              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ z              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ participant_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sequence_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sign           │ VARCHAR     │ YES     │ NULL    

In [19]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter


def landmark_motion_energy(group, window=7, polyorder=2):
    group = group.sort_values("frame")
    coords = group[["x", "y", "z"]].to_numpy()
    coords = pd.DataFrame(coords).interpolate(limit_direction="both").to_numpy()
    if len(coords) < window or np.isnan(coords).any():
        return np.nan
    smoothed = np.stack(
        [
            savgol_filter(coords[:, i], window_length=window, polyorder=polyorder)
            for i in range(3)
        ],
        axis=1,
    )
    velocity = np.diff(smoothed, axis=0)
    return np.sqrt(np.mean(np.sum(velocity**2, axis=1)))


# 1. Pick a sign and pull its rows out of DuckDB into pandas
SIGN = "book"  # replace with whatever gloss you want

sign_df = db.sql(f"""
    SELECT * FROM meta_holistic WHERE sign = '{SIGN}'
""").df()

print(f"{sign_df['path'].nunique()} videos found for sign '{SIGN}'")

# 2. Motion energy per (video, type, landmark_index) —
#    each video is its own trajectory, don't mix them
per_video_energy = (
    sign_df.groupby(["path", "type", "landmark_index"])
    .apply(landmark_motion_energy)
    .rename("motion_energy")
    .reset_index()
)

# 3. Aggregate across all videos of this sign, per landmark
sign_energy = (
    per_video_energy.groupby(["type", "landmark_index"])["motion_energy"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

sign_energy


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

KeyError: 'path'